#GCP Authentication & Configuration
Switch on notebook access in secrets as needed.

In [ ]:
import os
from google.colab import auth, userdata

# --- Authentication ---
auth.authenticate_user()

# --- GCP Configuration ---
PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]

os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID


assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

# Setup

In [ ]:
import json, time, hashlib
import pandas as pd
import requests
from datetime import datetime, timezone
from google.cloud import bigquery, storage

bq = bigquery.Client(project=PROJECT_ID, location=REGION)
gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET_NAME)

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)
    return bq.query(sql, job_config=config).to_dataframe()

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
MODEL = "era5"
HOURLY = ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "surface_pressure",
          "precipitation", "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m", "cloud_cover"]
RAW_PREFIX = f"raw/open_meteo/archive/model={MODEL}"
EXT_PREFIX = f"{RAW_PREFIX}/extended"
DATED = f"{PROJECT_ID}.tennis_clean.matches_dated"

editions = run(f"SELECT * FROM `{PROJECT_ID}.tennis_clean.tournament_editions` ORDER BY tourney_id")
print(len(editions), "editions in v1")

# Find editions that are too short
Compare each event's last possible match day with how far the fetched weather reaches.

In [ ]:
short_ids = run(f"""
SELECT m.tourney_id
FROM `{DATED}` m
JOIN `{PROJECT_ID}.tennis_clean.tournament_editions` e USING (tourney_id)
GROUP BY m.tourney_id
HAVING MAX(m.match_date_max) > ANY_VALUE(e.end_date)
""")["tourney_id"].tolist()

short_rows = editions[editions["tourney_id"].isin(short_ids)]
print(len(short_rows), "editions need a longer window")        # expect 6
short_rows[["tourney_id", "place", "tourney_level", "start_date", "end_date"]]

# Fetch the editions with a 14-day window
The original files are not touched. The new ones go into a separate extended/ folder, and re-running skips anything already saved.

In [ ]:
for row in short_rows.itertuples():
    blob = bucket.blob(f"{EXT_PREFIX}/{row.tourney_id}.json")
    if blob.exists():
        print(row.tourney_id, "already stored")
        continue
    new_end = (pd.Timestamp(row.start_date) + pd.Timedelta(days=14)).date()
    params = {"latitude": round(float(row.latitude), 4), "longitude": round(float(row.longitude), 4),
              "start_date": str(row.start_date), "end_date": str(new_end),
              "hourly": ",".join(HOURLY), "timezone": "auto", "models": MODEL}
    for attempt in range(5):
        r = requests.get(ARCHIVE_URL, params=params, timeout=60)
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 * 2 ** attempt)
            continue
        r.raise_for_status()
        break
    else:
        raise RuntimeError(f"Gave up on {row.tourney_id}")
    record = {"tourney_id": row.tourney_id, "request": {"url": ARCHIVE_URL, "params": params},
              "fetched_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
              "response": r.json()}
    blob.upload_from_string(json.dumps(record), content_type="application/json", if_generation_match=0)
    print(row.tourney_id, row.place, "fetched through", new_end)
    time.sleep(0.5)

In [ ]:
for row in short_rows.itertuples():
    blob = bucket.blob(f"{EXT_PREFIX}/{row.tourney_id}.json")
    if blob.exists():
        print(row.tourney_id, "already stored")
        continue
    new_end = (pd.Timestamp(row.start_date) + pd.Timedelta(days=14)).date()
    params = {"latitude": round(float(row.latitude), 4), "longitude": round(float(row.longitude), 4),
              "start_date": str(row.start_date), "end_date": str(new_end),
              "hourly": ",".join(HOURLY), "timezone": "auto", "models": MODEL}
    for attempt in range(5):
        r = requests.get(ARCHIVE_URL, params=params, timeout=60)
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 * 2 ** attempt)
            continue
        r.raise_for_status()
        break
    else:
        raise RuntimeError(f"Gave up on {row.tourney_id}")
    record = {"tourney_id": row.tourney_id, "request": {"url": ARCHIVE_URL, "params": params},
              "fetched_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
              "response": r.json()}
    blob.upload_from_string(json.dumps(record), content_type="application/json", if_generation_match=0)
    print(row.tourney_id, row.place, "fetched through", new_end)
    time.sleep(0.5)

# Validate and write the manifest

In [ ]:
entries = []
for b in gcs.list_blobs(BUCKET_NAME, prefix=f"{EXT_PREFIX}/"):
    data = b.download_as_bytes()
    hourly = json.loads(data)["response"]["hourly"]
    entries.append({"file": b.name, "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest(),
                    "n_hours": len(hourly["time"]),
                    "null_values": sum(v is None for k, vals in hourly.items() if k != "time" for v in vals)})
print(pd.DataFrame(entries)[["file", "n_hours", "null_values"]])     # expect 360 hours and 0 nulls each

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
bucket.blob(f"manifests/open_meteo_archive_era5_extended_{stamp}.json").upload_from_string(
    json.dumps({"endpoint": ARCHIVE_URL, "model": MODEL, "window": "start_date to +14 days", "files": entries}, indent=1),
    content_type="application/json")
print("Manifest written")

# Editions table (version 2)

In [ ]:
ed = run(f"SELECT * FROM `{PROJECT_ID}.tennis_clean.tournament_editions`").drop(columns=["n_matches"])
need = run(f"""SELECT tourney_id, ANY_VALUE(event_format) AS event_format,
                      MAX(match_date_max) AS need_until, COUNT(*) AS n_matches
               FROM `{DATED}` GROUP BY tourney_id""")

v2 = ed.merge(need, on="tourney_id", how="inner")    # inner join drops editions no longer in scope
for c in ["start_date", "end_date", "need_until"]:
    v2[c] = pd.to_datetime(v2[c])

short = v2["need_until"] > v2["end_date"]
base = f"{RAW_PREFIX}/"
v2["weather_path"] = base + v2["tourney_id"] + ".json"
v2.loc[short, "weather_path"] = base + "extended/" + v2.loc[short, "tourney_id"] + ".json"
v2.loc[short, "end_date"] = v2.loc[short, "start_date"] + pd.Timedelta(days=14)
assert (v2["need_until"] <= v2["end_date"]).all(), "some windows still outrun the weather"

stored = {b.name for b in bucket.list_blobs(prefix=base)}
print("editions:", len(v2), "| extended window:", int(short.sum()),
      "| missing files:", sum(p not in stored for p in v2["weather_path"]))   # expect 872 | 6 | 0

out = v2.drop(columns=["need_until"]).copy()
out["start_date"] = out["start_date"].dt.date
out["end_date"] = out["end_date"].dt.date
bq.load_table_from_dataframe(out, f"{PROJECT_ID}.tennis_clean.tournament_editions_v2",
                             job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")).result()

csv_text = out.to_csv(index=False)
blob = bucket.blob("reference/editions/v2/tournament_editions.csv")
if blob.exists():
    print("v2 file already exists, not overwriting")
else:
    blob.upload_from_string(csv_text, content_type="text/csv", if_generation_match=0)
    print("Saved v2, sha256", hashlib.sha256(csv_text.encode()).hexdigest()[:12])